<b><font size="6" color="#E8800A">Week 5 · Performance Measures</font></b><br>

A model that runs is not yet a model that works: we also need to know **how well** it predicts. **Performance measures** quantify that, so that models can be compared and one chosen for the problem at hand. This class covers the most common measures for **regression** and for **classification**.

<div class="alert alert-block alert-info">

## Table of Contents<a class="anchor" id="toc"></a>
### [<font color='#E8800A'>1 - Regression Problems</font>](#regression)
* [<font color='#E8800A'>1.1. - R² Score</font>](#rsquare)
* [<font color='#E8800A'>1.2. - Adjusted R² Score</font>](#adjusted)
* [<font color='#E8800A'>1.3. - MAE</font>](#mae)
* [<font color='#E8800A'>1.4. - MSE and RMSE</font>](#mse)
* [<font color='#E8800A'>1.5. - MedAE</font>](#medae)
* [<font color='#E8800A'>1.6. - MAPE</font>](#mape)
* [<font color='#E8800A'>1.7. - Comparing Regression Metrics</font>](#comparison)
### [<font color='#E8800A'>2 - Classification Problems</font>](#classification)
* [<font color='#E8800A'>2.1. - The Confusion Matrix</font>](#confusion)
* [<font color='#E8800A'>2.2. - The Accuracy Score</font>](#accuracy)
* [<font color='#E8800A'>2.3. - The Precision</font>](#precision)
* [<font color='#E8800A'>2.4. - The Recall</font>](#recall)
* [<font color='#E8800A'>2.5. - The F1 Score</font>](#f1)
* [<font color='#E8800A'>2.6. - The F-beta Score</font>](#fbeta)
* [<font color='#E8800A'>2.7. - Classification Thresholds</font>](#thresholds)
* [<font color='#E8800A'>2.8. - ROC Curve and AUC Score</font>](#roc)
* [<font color='#E8800A'>2.9. - Precision-Recall Curve</font>](#pr-curve)
* [<font color='#E8800A'>2.10. - Comparing Classification Metrics</font>](#classification-comparison)
* [<font color='#E8800A'>Key takeaways</font>](#takeaways)
### [<font color='#E8800A'>3 - Multiclass Classification (Extra)</font>](#multiclass)
* [<font color='#E8800A'>3.1. - Multiclass Confusion Matrix</font>](#multiclass-confusion)
* [<font color='#E8800A'>3.2. - Macro-Averaged Metrics</font>](#macro)
* [<font color='#E8800A'>3.3. - Weighted-Averaged Metrics</font>](#weighted)
* [<font color='#E8800A'>3.4. - Micro-Averaged Metrics</font>](#micro)
</div>

<a class="anchor" id="regression">

## <font color='#E8800A'>1. Regression Problems</font>
</a>

Every regression metric below measures how far predictions land from the truth: either the share of the target's variance a model explains, or the size of its misses. They need predictions to score, so we start by fitting a regression model.

__`Step 1`__ Import the needed libraries

[Back to TOC](#toc)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

#sklearn libraries
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

from sklearn.metrics import classification_report

RANDOM_STATE = 33
np.random.seed(RANDOM_STATE)  # For reproducibility

__`Step 2`__ Import __cars4you.csv__ and the inherited
Week 4 recipe. Keep the raw predictors as **data_cars** until after the split,
with **target_cars** as `price`.

Run this notebook from its own folder inside the course
repository: that is what makes the `data/` paths below work. If you
downloaded this file on its own from Moodle, move it into the repository before
you run it.

### The inherited recipes

This notebook evaluates both course spines, so each half loads its own recipe.
Four decisions arrive from the log: the column roles and
fill rules Week 3 recorded, the transform it measured, the encoder and scaler
Week 4 selected, and the selection strategy Week 4 exported. `preprocessing.py`
rebuilds them from the log, so what this notebook applies is whatever those
weeks decided.

In [ ]:
import sys
from pathlib import Path
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from preprocessing import (
    PreparedEstimator,
    classification_preprocessor,
    load_classification,
    load_regression,
    regression_preprocessor,
)
CLASSIFICATION_RECIPE_LOG = "../../logs/week_04_feature_work_classification_log.json"
REGRESSION_RECIPE_LOG = "../../logs/week_04_feature_work_regression_log.json"

In [ ]:
cars, regression_recipe = load_regression(
    "../../data/interim/cars4you.csv",
    REGRESSION_RECIPE_LOG,
)
data_cars = cars.drop(columns=["price", "model"])
target_cars = cars["price"]

print(f"{data_cars.shape[0]:,} rows x {data_cars.shape[1]} raw predictors")
print(f"inherited: {regression_recipe['encoding']} + "
      f"{regression_recipe['scaler']}, log1p {regression_recipe['log1p']}")

__`Step 3`__ Use the method **train_test_split** from sklearn.model_selection to split your dataset between train (80%) and validation (20%).

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(data_cars, 
                                                    target_cars, 
                                                    test_size=0.2, 
                                                    random_state=RANDOM_STATE, 
                                                    shuffle=True, 
                                                   )

__`Step 4`__ Create an instance of LinearRegression named `lr` with the default parameters, paired with the inherited recipe in a `PreparedEstimator`, and fit it to your training data.

In [ ]:
lr = PreparedEstimator(
    regression_preprocessor(regression_recipe),
    LinearRegression(),
).fit(X_train, y_train)

__`Step 5`__ Predict with `lr` on both sets, assigning the predictions to `y_pred_train` and `y_pred_val`.

In [ ]:
y_pred_train = lr.predict(X_train)
y_pred_val = lr.predict(X_val)

__`Step 6`__ From __sklearn.metrics__ import `r2_score`, `mean_absolute_error`, `mean_squared_error`, `root_mean_squared_error`, `median_absolute_error` and `mean_absolute_percentage_error`. This notebook imports each tool where it is first used; in a project, every import goes at the top.

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, median_absolute_error, root_mean_squared_error, mean_absolute_percentage_error

<a class="anchor" id="rsquare">

### <font color='#E8800A'>1.1. R² Score</font>

</a>

$R^{2}$, the coefficient of determination, is the share of the target's variance that the model explains. A perfect fit scores 1, predicting the average target for every row scores 0, and a model that does worse than that average scores below 0.

$$
R^{2} = 1 - \frac{SS_{res}}{SS_{tot}}
$$
Where:
- $SS_{res} = \sum (y_i - \hat{y}_i)^2$ is the sum of squared residuals, the gaps between observed and predicted values;
- $SS_{tot} = \sum (y_i - \bar{y})^2$ is the total sum of squares, the gaps between observed values and their mean.

<div class="alert alert-block alert-info">
<a href = 'https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html#sklearn.metrics.r2_score'>sklearn.metrics.r2_score(y_true, y_pred, ... )</a>

__Definition:__ <br>
$R^2$ (coefficient of determination) regression score function.

__Interpretation:__ <br>
Best possible score is 1.0 and it can be negative (because the model can be arbitrarily worse). 

__Parameters:__ <br>
_y_true_: Ground truth (correct) target values; <br>
_y_pred_: Estimated target values; <br>
...
</div>

__`Step 7`__ Score `lr` with $R^2$ on both sets. For every metric in this notebook, the training score is computed for you and you write the validation score.

In [ ]:
r2_t = r2_score(y_train, y_pred_train)
r2_t

In [ ]:
r2_v = ...  # <-- CODE HERE
...  # <-- CODE HERE


**Advantages of $R^2$**:
- Unit-free: it reads as a share of variance, so it compares targets measured in different units.
- Familiar: it is the standard summary of a regression fit.

**Disadvantages of $R^2$**:
- On the training rows of a linear regression it never falls when a predictor is added, so it cannot compare models with different numbers of features there.
- It says nothing about the size of an error in the target's own units.
- A high $R^2$ does not rule out a systematic bias, such as a model that under-predicts every expensive car.

__When should $R^2$ be used?__ <br>
- To report how much of the target's variation a model explains.
- To compare models of the same target, beside an error metric in the target's units.

<a class="anchor" id="adjusted">

### <font color='#E8800A'>1.2. Adjusted R² Score</font>

</a>

Adding a predictor never lowers a linear regression's training $R^2$, even when the predictor is noise. **Adjusted $R^2$**, $\bar{R}^2$, charges for each predictor relative to the sample size, so it rises only when a predictor improves the fit by more than chance would. scikit-learn has no function for it, so we compute it:

$$
\bar{R}^2 = 1 - \left(1 - R^2\right) \cdot \frac{n - 1}{n - p - 1}
$$

where $n$ is the number of rows and $p$ the number of predictors, here the columns the recipe hands the model.

__`Step 8`__ Compute $\bar{R}^2$ for both sets.

In [ ]:
r2 = r2_score(y_train, y_pred_train)
n = len(y_train)
p = lr.named_steps['prepare'].transform(X_train).shape[1]


def adjusted_r_squared(r_squared, sample_size, feature_count):
    """Adjust R squared for sample size and fitted feature count."""
    return 1 - (1 - r_squared) * (sample_size - 1) / (
        sample_size - feature_count - 1
    )


ar2_t = adjusted_r_squared(r2, n, p)
ar2_t

In [ ]:
r2 = ...  # <-- CODE HERE
n = ...  # <-- CODE HERE
p = ...  # <-- CODE HERE

ar2_v = ...  # <-- CODE HERE
...  # <-- CODE HERE


**Advantages of $\bar{R}^2$**:
- Penalises predictors that do not earn their place, so it can compare models with different numbers of features.
- Keeps the reading of $R^2$ as a share of variance explained.

**Disadvantages of $\bar{R}^2$**:
- Its penalty assumes a linear model with $p$ coefficients; for trees and other flexible models the number of columns is not the model's complexity.
- Like $R^2$, it gives no error in the target's units.

__When should $\bar{R}^2$ be used?__ <br>
- When comparing linear regressions built on different sets of features.

<a class="anchor" id="mae">

### <font color='#E8800A'>1.3. MAE (Mean Absolute Error)</font>

</a>

The error metrics that follow measure how far predictions land from the truth. The most direct is the **Mean Absolute Error (MAE)**: the average size of a miss, in the target's own units.

$$
MAE = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|
$$

__`Step 9`__ Check the MAE of `lr` on both sets.

<div class="alert alert-block alert-info">
<a href = 'https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html#sklearn.metrics.mean_absolute_error'>sklearn.metrics.mean_absolute_error(y_true, y_pred, ... )</a>

__Definition:__ <br>
Mean absolute error regression loss.

__Interpretation:__ <br>
Best possible value is 0.0. MAE is always non-negative.

__Parameters:__ <br>
_y_true_: Ground truth (correct) target values; <br>
_y_pred_: Estimated target values; <br>
...
</div>

In [ ]:
mae_t = mean_absolute_error(y_train, y_pred_train)
mae_t

In [ ]:
mae_v = ...  # <-- CODE HERE
...  # <-- CODE HERE


**Advantages of MAE**:
- In the target's units: an MAE of 1,000 on a price means the model misses by 1,000 EUR on average.
- Every unit of error counts the same, so one large miss does not dominate it.

**Disadvantages of MAE**:
- For the same reason, it cannot tell many small misses from a few large ones: both can give the same MAE.
- Scale-dependent: it cannot compare targets measured in different units.

__When should MAE be used?__ <br>
- When the cost of an error grows in proportion to its size.
- When the number has to be explained to someone outside the team.

<a class="anchor" id="mse">

### <font color='#E8800A'>1.4. MSE (Mean Squared Error) and RMSE (Root Mean Squared Error)</font>

</a>

The **Mean Squared Error** squares each miss, so a large miss counts for more than several small ones; its square root, the **RMSE**, returns to the target's units:

$$
MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 \qquad RMSE = \sqrt{MSE}
$$

__`Step 10`__ Check the MSE and RMSE of `lr` on both sets.

### MSE

<div class="alert alert-block alert-info">
<a href = 'https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html'>sklearn.metrics.mean_squared_error(y_true, y_pred, ... )</a>

__Definition:__ <br>
Mean squared error regression loss.

__Interpretation:__ <br>
Best possible value is 0.0. MSE is always non-negative.

__Parameters:__ <br>
_y_true_: Ground truth (correct) target values; <br>
_y_pred_: Estimated target values; <br>
...
</div>

In [ ]:
mse_t = mean_squared_error(y_train, y_pred_train)
mse_t

In [ ]:
mse_v = ...  # <-- CODE HERE
...  # <-- CODE HERE


### RMSE

<div class="alert alert-block alert-info">
<a href = 'https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html'>sklearn.metrics.root_mean_squared_error(y_true, y_pred, ... )</a>

__Definition:__ <br>
Root mean squared error regression loss.

__Interpretation:__ <br>
Best possible value is 0.0. RMSE is always non-negative.

__Parameters:__ <br>
_y_true_: Ground truth (correct) target values; <br>
_y_pred_: Estimated target values; <br>
...
</div>

In [ ]:
rmse_t = root_mean_squared_error(y_train, y_pred_train)
rmse_t

In [ ]:
rmse_v = ...  # <-- CODE HERE
...  # <-- CODE HERE


**Advantages of MSE and RMSE**:
- Large errors weigh more than small ones, which suits problems where a big miss is disproportionately costly.
- MSE is smooth everywhere, which is why least-squares fitting and many training losses minimise it.
- RMSE is back in the target's units.

**Disadvantages of MSE and RMSE**:
- One large miss can dominate the score, so a few outliers can make an otherwise good model look poor.
- MSE is in squared units, which few readers can interpret; RMSE fixes the units but not the weighting.

__When should MSE or RMSE be used?__ <br>
- When large errors are disproportionately costly.
- MSE and RMSE always rank models in the same order, so MSE suits optimisation and RMSE suits reporting.

<a class="anchor" id="medae">

### <font color='#E8800A'>1.5. MedAE (Median Absolute Error)</font>

</a>

The **Median Absolute Error** is the median of the absolute errors rather than their mean, so the few largest misses cannot move it: it reports the typical miss.

$$
MedAE = \text{median}(|y_1 - \hat{y}_1|, |y_2 - \hat{y}_2|, \ldots, |y_n - \hat{y}_n|)
$$

__`Step 11`__ Check the MedAE of `lr` on both sets.

<div class="alert alert-block alert-info">
<a href = 'https://scikit-learn.org/stable/modules/generated/sklearn.metrics.median_absolute_error.html'>sklearn.metrics.median_absolute_error(y_true, y_pred, ... )</a>

__Definition:__ <br>
Median absolute error regression loss

__Interpretation:__ <br>
Best possible value is 0.0. MedAE is always non-negative.

__Parameters:__ <br>
_y_true_: Ground truth (correct) target values; <br>
_y_pred_: Estimated target values; <br>
...
</div>

In [ ]:
medae_t = median_absolute_error(y_train, y_pred_train)
medae_t

In [ ]:
medae_v = ...  # <-- CODE HERE
...  # <-- CODE HERE


**Advantages of MedAE**:
- Unmoved by the few largest errors, so it reports the typical miss when a handful of predictions go badly wrong.
- In the target's units, like MAE.

**Disadvantages of MedAE**:
- Blind to the largest errors: the half of the misses above the median can grow without limit and the MedAE does not change.
- Less often reported, so it is harder to compare with published results.

__When should MedAE be used?__ <br>
- Beside MAE or RMSE, to show whether a few large misses are driving them.

<a class="anchor" id="mape">

### <font color='#E8800A'>1.6. MAPE (Mean Absolute Percentage Error)</font>

</a>

The **Mean Absolute Percentage Error** divides each miss by the true value, so it is unit-free and compares models of different targets: a MAPE of 0.10 means a miss of 10% of the true value on average.

$$
MAPE = \frac{1}{n} \sum_{i=1}^{n} \left|\frac{y_i - \hat{y}_i}{y_i}\right|
$$

__`Step 12`__ Check the MAPE of `lr` on both sets.

<div class="alert alert-block alert-info">
<a href = 'https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_percentage_error.html#sklearn.metrics.mean_absolute_percentage_error'>sklearn.metrics.mean_absolute_percentage_error(y_true, y_pred, ... )</a>

__Definition:__ <br>
Mean absolute percentage error (MAPE) regression loss.

__Interpretation:__ <br>
Best possible value is 0.0. scikit-learn returns a fraction rather than a percentage: 0.05 means 5%.

__Parameters:__ <br>
_y_true_: Ground truth (correct) target values; <br>
_y_pred_: Estimated target values; <br>
...
</div>

In [ ]:
mape_t = mean_absolute_percentage_error(y_train, y_pred_train)
mape_t

In [ ]:
mape_v = ...  # <-- CODE HERE
...  # <-- CODE HERE


**Advantages of MAPE**:
- Unit-free, so it compares across targets and datasets.
- Easy to communicate: "off by 10% on average".

**Disadvantages of MAPE**:
- Undefined when a true value is zero, and explosive when true values are close to zero.
- Asymmetric: an under-prediction can miss by at most 100% of a positive true value, while an over-prediction has no ceiling, so minimising MAPE favours predictions that are too low.
- Meaningless for a target that changes sign.

__When should MAPE be used?__ <br>
- When the target is always positive and well away from zero, as a price is, and the relative error is what matters.

## <font color='#E8800A'>Comparing Differences</font>

[Back to TOC](#toc)


In [ ]:
# Every metric on both sets, computed from the predictions.
features = lr.named_steps['prepare'].transform(X_train).shape[1]
scores = {'R2': r2_score, 'MAE': mean_absolute_error, 'MSE': mean_squared_error,
          'RMSE': root_mean_squared_error, 'MedAE': median_absolute_error,
          'MAPE': mean_absolute_percentage_error}
train_scores = {name: score(y_train, y_pred_train) for name, score in scores.items()}
val_scores = {name: score(y_val, y_pred_val) for name, score in scores.items()}
train_scores['Adjusted R2'] = adjusted_r_squared(train_scores['R2'], len(y_train), features)
val_scores['Adjusted R2'] = adjusted_r_squared(val_scores['R2'], len(y_val), features)
regression_metrics = pd.DataFrame({'Train': train_scores, 'Validation': val_scores})
regression_metrics

<a class="anchor" id="comparison">

### <font color='#E8800A'>1.7. How Metrics Can Change Model Selection: A Numerical Example</font>

</a>

Each metric above weighs errors differently, so they need not agree on which model is best. A small constructed example with three hypothetical models shows how far they can disagree.

__`Step 13`__ Define the true prices of 6 houses, in thousands of dollars, and the predictions of three hypothetical models:

* **`preds_model_A`**: off by a small, consistent amount on every house.
* **`preds_model_B`**: almost perfect, except for one very large error.
* **`preds_model_C`**: a mix of perfect predictions and moderate errors, none as extreme as Model B's.

In [ ]:
import numpy as np

# Actual values
y_true = np.array([100, 150, 200, 250, 300, 350])

# Model A: Consistent, small errors
preds_model_A = np.array([110, 160, 210, 260, 310, 360])

# Model B: Mostly perfect, one large error
preds_model_B = np.array([100, 150, 200, 250, 300, 450])

# Model C: A mix of perfect and moderate errors
preds_model_C = np.array([100, 150, 225, 275, 300, 350])

__`Step 14`__ Write a helper that computes every regression metric for one set of predictions.

In [ ]:
def regression_metrics(y_true, y_pred):
    """Return the regression metrics used in this comparison."""
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    medae = median_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return [r2, mae, mse, rmse, medae, mape]

__`Step 15`__ Use our function to compute the metrics for each of the three models.

In [ ]:
metrics_A = regression_metrics(y_true, preds_model_A)
metrics_B = regression_metrics(y_true, preds_model_B)
metrics_C = regression_metrics(y_true, preds_model_C)

__`Step 16`__ Finally, organize these results into a DataFrame to make them easy to compare.

In [ ]:
comparison_df = pd.DataFrame({
    'Metric': ['R2', 'MAE', 'MSE', 'RMSE', 'MedAE', 'MAPE'],
    'Model A': metrics_A,
    'Model B': metrics_B,
    'Model C': metrics_C
})

comparison_df

__`Step 17`__ Analyze the results and see what they tell us.

<div class="alert alert-block alert-success">

### Analysis of the Results

The "best" model changes with the metric:

* **Model A** (small, consistent errors) is best on $R^2$ (0.98), $MSE$ (100) and $RMSE$ (10).
* **Model B** (mostly perfect, one large error) ties with C for the best $MedAE$ (0) and is by far the worst on $MSE$ (1666.7) and $RMSE$ (40.8).
* **Model C** (a milder version of B) is best on $MAE$ (8.33) and $MAPE$ (0.0375), and ties with B on $MedAE$.

### What to take from this comparison

Choose the metric from the cost of an error in your problem, before you see the scores.
</div>

<a class="anchor" id="classification">

## <font color='#E8800A'>2. Classification Problems</font>
</a>

A categorical target restores the notion of a correct answer, so distance gives way to counting: a prediction either matches the label or it does not.

[Back to TOC](#toc)


__`Step 18`__ Import the needed libraries to apply Logistic Regression.

In [ ]:
from sklearn.linear_model import LogisticRegression

__`Step 19`__ Import the dataset __champions.csv__, the
cleaned classification spine, and define the features as __data_champions__ and
`Outcome` as __target_champions__.

In [ ]:
champions, classification_recipe = load_classification(
    "../../data/interim/champions.csv",
    CLASSIFICATION_RECIPE_LOG,
)
target_champions = champions["Outcome"]
champions_features = champions.drop(
    columns=["Outcome", "RecordID", "Athlete Id"]
)
data_champions = champions_features
categorical_columns = classification_recipe["categorical"]
numeric_columns = classification_recipe["numeric"]

print(f"{len(categorical_columns)} categorical + {len(numeric_columns)} numeric"
      f" raw columns for {len(champions_features):,} athletes")
print(f"inherited: {classification_recipe['encoding']} + "
      f"{classification_recipe['scaler']}; win rate {target_champions.mean():.4f}")

__`Step 20`__ Use `train_test_split` from `sklearn.model_selection` to split your dataset into train (80%) and validation (20%).

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(data_champions, 
                                                  target_champions, 
                                                  test_size = 0.2, 
                                                  random_state=RANDOM_STATE, 
                                                  stratify = target_champions) #in this case, we use stratify to keep the same proportion of classes in train and validation sets

__`Step 21`__ Fit the inherited recipe, taking every learned
value from the **training set only**.

In [ ]:
# Build one UNFITTED copy of the Week 4 recipe
training_rows = len(X_train)
classification_prepare = classification_preprocessor(classification_recipe)
X_train = classification_prepare.fit_transform(X_train, y_train)
X_val = classification_prepare.transform(X_val)

print(f"{len(classification_prepare.encoded_names_)} encoded columns,"
      f" of which {X_train.shape[1]} survive the inherited selection")
print(f"recipe fitted on {training_rows:,} training rows")

__`Step 22`__ Confirm that the inherited scaler was fitted with
the rest of the recipe on the **training set only**.

In [ ]:
scaler = classification_prepare.scaler_
print(f"{type(scaler).__name__}, the scaler the Week 4 log carries")
print(f"fitted on {training_rows:,} training rows across"
      f" {scaler.n_features_in_} numeric columns;"
      f" validation contributed 0 rows")

__`Step 23`__ Create an instance of `LogisticRegression` named `log_model` with the default parameters and fit it to your training data.

In [ ]:
log_model = LogisticRegression()

In [ ]:
log_model.fit(X_train, y_train)

__`Step 24`__ Once the model is trained, obtain predictions for both the training and validation sets.

In [ ]:
y_pred_train = log_model.predict(X_train)
y_pred_val = log_model.predict(X_val)

__`Step 25`__ From __sklearn.metrics__ import `confusion_matrix`, `accuracy_score`, `precision_score`, `recall_score`, `f1_score`, `roc_auc_score`, `precision_recall_curve` and `roc_curve`.

Most classification metrics are computed from the **confusion matrix**, the table that counts each kind of correct and incorrect prediction.

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, precision_recall_curve, roc_curve

<a class="anchor" id="confusion">

### <font color='#E8800A'>2.1. The Confusion Matrix</font>

</a>

The confusion matrix counts a classifier's predictions against the true labels: rows are the actual classes and columns the predicted ones. For two classes it has four cells:

|                      | Predicted Negative | Predicted Positive |
|----------------------|--------------------|--------------------|
| **Actual Negative**   | True Negative (TN) | False Positive (FP)|
| **Actual Positive**   | False Negative (FN)| True Positive (TP) |

<div class="alert alert-block alert-info">
<a href = 'https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html#sklearn.metrics.confusion_matrix'>sklearn.metrics.confusion_matrix(y_true, y_pred, ...)</a>

__Definition:__ <br>
Compute confusion matrix to evaluate the accuracy of a classification

__Parameters:__ <br>
_y_true_: Ground truth (correct) target values.; <br>
_y_pred_: Estimated targets as returned by a classifier.; <br>
...
</div>

__`Step 26`__ Obtain the confusion matrix.

In [ ]:
cm_train = confusion_matrix(y_train, y_pred_train)
cm_train

In [ ]:
cm_val = ...  # <-- CODE HERE
...  # <-- CODE HERE


**Understanding the Confusion Matrix**

scikit-learn prints the confusion matrix in this layout:

```
[[TN  FP]
 [FN  TP]]
```

- **TN**: negatives predicted negative
- **FP**: negatives predicted positive (a false alarm, or Type I error)
- **FN**: positives predicted negative (a miss, or Type II error)
- **TP**: positives predicted positive

**Advantages of the Confusion Matrix**:
- Shows every kind of outcome, so it tells false alarms apart from misses.
- Every metric in this section is computed from it.

**Disadvantages of the Confusion Matrix**:
- Four numbers, not one, so it cannot rank models on its own.
- Raw counts depend on the class sizes, so matrices from datasets of different balance do not compare directly.

**When should the Confusion Matrix be used?**
- Always, before any single-number metric, to see which errors the model makes.

<a class="anchor" id="accuracy">

### <font color='#E8800A'>2.2. The Accuracy Score</font>

</a>

**Accuracy** is the share of predictions that are correct, positive and negative alike.

$$
Accuracy = \frac{TP + TN}{TP + TN + FP + FN}
$$

<div class="alert alert-block alert-info">
<a href = 'https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html#sklearn.metrics.accuracy_score'>sklearn.metrics.accuracy_score(y_true, y_pred, normalize=True,...)</a>

__Definition:__ <br>
Accuracy classification score.

__Interpretation:__ <br>
If normalize is True, then the best performance is 1. When normalize = False, then the best performance is the number of samples.

__Parameters:__ <br>
_y_true_: Ground truth (correct) target values.; <br>
_y_pred_: Estimated targets as returned by a classifier.; <br>
_normalize_: If False, return the number of correctly classified samples. Otherwise, return the fraction of correctly classified samples. <br>
...
</div>

__`Step 27`__ Compute the accuracy on both sets. As in the regression half, the training score is computed for you and you write the validation score.

In [ ]:
accuracy_train = accuracy_score(y_train, y_pred_train)
accuracy_train

In [ ]:
accuracy_val = ...  # <-- CODE HERE
...  # <-- CODE HERE


**Understanding Accuracy**

A screening test on a thousand people, ten of them actually sick:

| | Predicted not sick | Predicted sick | total |
|---|---:|---:|---:|
| Actually not sick | **990 (TN)** | **0 (FP)** | 990 |
| Actually sick | **9 (FN)** | **1 (TP)** | 10 |
| total | 999 | 1 | 1,000 |

Accuracy reads **every cell**:

$$\text{accuracy} = \frac{TP + TN}{TP + TN + FP + FN} = \frac{1 + 990}{1{,}000} = 0.991$$

An accuracy of 99.1%, on a test that found one sick person in ten.

**Advantages of Accuracy**:
- Simple to compute and to explain.
- Counts both classes, so it summarises the whole confusion matrix in one number.

**Disadvantages of Accuracy**:
- Misleading when one class is rare: predicting the majority class for everyone scores the majority's share, as the test above shows.
- Treats a false alarm and a miss as equally costly.

**When should Accuracy be used?**
- When the classes are roughly balanced and the two kinds of error cost about the same.

<a class="anchor" id="precision">

### <font color='#E8800A'>2.3. The Precision</font>

</a>

**Precision** answers: of the rows the model called positive, how many were positive?

$$
Precision = \frac{TP}{TP + FP}
$$

<div class="alert alert-block alert-info">
<a href = 'https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html#sklearn.metrics.precision_score'>sklearn.metrics.precision_score(y_true, y_pred, ...)</a>

__Definition:__ <br>
Compute the precision.

__Interpretation:__ <br>
The best value is 1, and the worst value is 0.

__Parameters:__ <br>
_y_true_: Ground truth (correct) target values.; <br>
_y_pred_: Estimated targets as returned by a classifier.; <br>
...
</div>

__`Step 28`__ Compute the precision score for both the training and validation sets.

In [ ]:
precision_train = precision_score(y_train, y_pred_train)
precision_train

In [ ]:
precision_val = ...  # <-- CODE HERE
...  # <-- CODE HERE


**Understanding Precision**

The same screening test:

| | Predicted not sick | Predicted sick | total |
|---|---:|---:|---:|
| Actually not sick | 990 (TN) | **0 (FP)** | 990 |
| Actually sick | 9 (FN) | **1 (TP)** | 10 |
| total | 999 | 1 | 1,000 |

Precision reads **one column**, the one the model called positive:

$$\text{precision} = \frac{TP}{TP + FP} = \frac{1}{1 + 0} = 1.000$$

Perfect precision, on a test that found one sick person in ten.

**Advantages of Precision**:
- Says how far a positive prediction can be trusted.
- The right measure when acting on a positive is expensive.

**Disadvantages of Precision**:
- Ignores the positives the model missed: calling only the single most certain row positive can score 1.
- Ignores true negatives.

**When should Precision be used?**
- When a false positive is the costly error. In **spam filtering** a false positive sends a legitimate email to the spam folder, where it may never be read; in **fraud blocking** it stops a legitimate payment.

<a class="anchor" id="recall">

### <font color='#E8800A'>2.4. The Recall</font>

</a>

**Recall**, also called **sensitivity** or the **true positive rate**, answers: of the rows that were positive, how many did the model find?

$$
Recall = \frac{TP}{TP + FN}
$$

<div class="alert alert-block alert-info">
<a href = 'https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html'>sklearn.metrics.recall_score(y_true, y_pred, ...)</a>

__Definition:__ <br>
Compute the recall.

__Interpretation:__ <br>
The best value is 1 and the worst value is 0.

__Parameters:__ <br>
_y_true_: Ground truth (correct) target values.; <br>
_y_pred_: Estimated targets as returned by a classifier.; <br>
...
</div>

__`Step 29`__ Compute the recall score for both the training and validation sets.

In [ ]:
recall_train = recall_score(y_train, y_pred_train)
recall_train

In [ ]:
recall_val = ...  # <-- CODE HERE
...  # <-- CODE HERE


**Understanding Recall**

The same screening test:

| | Predicted not sick | Predicted sick | total |
|---|---:|---:|---:|
| Actually not sick | 990 (TN) | 0 (FP) | 990 |
| Actually sick | **9 (FN)** | **1 (TP)** | 10 |
| total | 999 | 1 | 1,000 |

Recall reads **one row**, the one that was actually positive:

$$\text{recall} = \frac{TP}{TP + FN} = \frac{1}{1 + 9} = 0.100$$

The same counts that gave 99.1% accuracy and perfect precision give 10% recall.

**Advantages of Recall**:
- Says what share of the actual positives the model finds.
- The right measure when missing a positive is expensive.

**Disadvantages of Recall**:
- Ignores false alarms: predicting positive for everyone scores 1.
- Ignores true negatives.

**When should Recall be used?**
- When a false negative is the costly error. In **disease screening** a missed patient goes untreated and may infect others, while a false alarm costs a second test.

<a class="anchor" id="f1">

### <font color='#E8800A'>2.5. The F1 Score</font>

</a>

The **F1 score** is the harmonic mean of precision and recall: one number that is high only when both are.

$$
F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall} = \frac{2 \times TP}{2 \times TP + FP + FN}
$$

The harmonic mean is pulled towards the smaller of the two, so a model cannot buy a high F1 with perfect recall and poor precision, or the reverse.

<div class="alert alert-block alert-info">
<a href = 'https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html#sklearn.metrics.f1_score'>sklearn.metrics.f1_score(y_true, y_pred, ...)</a>

__Definition:__ <br>
Compute the F1 score, also known as balanced F-score or F-measure.

__Interpretation:__ <br>
F1 score reaches its best value at 1 and worst score at 0.

__Parameters:__ <br>
_y_true_: Ground truth (correct) target values.; <br>
_y_pred_: Estimated targets as returned by a classifier.; <br>
...
</div>

__`Step 30`__ Compute the F1 score for both the training and validation sets.

In [ ]:
f1_train = f1_score(y_train, y_pred_train)
f1_train

In [ ]:
f1_val = ...  # <-- CODE HERE
...  # <-- CODE HERE


In [ ]:
# The F1 any validation score has to beat: a win for every row.
always_win = np.ones(len(y_val), dtype=int)
print(f"F1 of predicting a win for every validation row: {f1_score(y_val, always_win):.4f}")

**Advantages of F1 Score**:
- One number that needs both precision and recall to be good.
- More informative than accuracy when the positive class is the rare one.

**Disadvantages of F1 Score**:
- Weighs a false positive and a false negative equally, whatever they cost.
- Ignores true negatives, so it depends on which class is called positive. When the positive class is the majority, as a win is here, predicting a win for every athlete already scores the high F1 printed above.

**When should F1 Score be used?**
- When both kinds of error matter, neither clearly dominates, and one number is needed to compare models.

<a class="anchor" id="fbeta">

### <font color='#E8800A'>2.6. The F-beta Score</font>

</a>

F1 weighs a false positive and a false negative equally. $F_\beta$ generalises it
with a weight you choose, and F1 is its case $\beta = 1$:

$$F_\beta = (1 + \beta^2)\cdot\frac{\text{precision}\cdot\text{recall}}{\beta^2\cdot\text{precision} + \text{recall}}$$

$\beta$ is how many times more important recall is than precision:

- $\beta = 1$ weighs them equally, which is F1.
- $\beta = 2$ treats recall as twice as important. Use it when a **missed
  positive** is the expensive error.
- $\beta = 0.5$ treats precision as twice as important. Use it when a **false
  alarm** is the expensive error.

Choose $\beta$ from the cost of the two errors in your problem, before you see
any scores.

<div class="alert alert-block alert-info">
<a href = 'https://scikit-learn.org/stable/modules/generated/sklearn.metrics.fbeta_score.html'>sklearn.metrics.fbeta_score(y_true, y_pred, *, beta, ... )</a>

__Definition:__ <br>
The weighted harmonic mean of precision and recall.

__Interpretation:__ <br>
Best possible value is 1.0, worst is 0.0. `beta < 1` favours precision,
`beta > 1` favours recall.

__Parameters:__ <br>
_y_true_: Ground truth (correct) target values; <br>
_y_pred_: Estimated target values; <br>
_beta_: Weight of recall relative to precision; <br>
...
</div>

__`Step 31`__ Score the same predictions at three values of $\beta$,
next to the precision and recall they are built from.

In [ ]:
from sklearn.metrics import fbeta_score

print(f"precision {precision_score(y_val, y_pred_val):.4f}"
      f"   recall {recall_score(y_val, y_pred_val):.4f}")
for beta in [0.5, 1, 2]:
    print(f"F{beta:<4} = {fbeta_score(y_val, y_pred_val, beta=beta):.4f}")

Precision is **0.8033** and recall **0.8965**, so this model finds most
of the athletes who went on to win and pays for it with false alarms. Every
$F_\beta$ sits between those two numbers and moves towards whichever one it
weighs more: **0.8204** at $\beta = 0.5$, **0.8474** at $\beta = 1$ and
**0.8762** at $\beta = 2$.

<a class="anchor" id="thresholds">

### <font color='#E8800A'>2.7. Classification Thresholds</font>

</a>

A logistic regression does not output a label. It outputs a **probability** that the row is positive, and the label comes from comparing that probability with a **threshold**: at or above it, the row is predicted positive. scikit-learn's `predict` uses 0.5.

Moving the threshold trades one error for the other. Take five patients:

| Patient | Probability | Actually sick? |
|---|---:|---|
| A | 0.95 | yes |
| B | 0.65 | yes |
| C | 0.45 | no |
| D | 0.30 | yes |
| E | 0.10 | no |

At 0.5, A and B are called sick and D is missed: recall 2/3. At 0.3, D is caught and recall is 3/3, while C becomes a false alarm. A **lower** threshold calls more rows positive, which raises recall and lowers precision; a **higher** one does the opposite.

[Back to TOC](#toc)

__`Step 32`__ Get the predicted probability of a win for every validation row.

In [ ]:
# Get probability predictions for the positive class
y_pred_proba_val = log_model.predict_proba(X_val)[:, 1]

# Show first 10 probabilities
print("First 10 probability predictions:")
print(y_pred_proba_val[:10])
print("\nActual labels:")
print(y_val.values[:10])

__`Step 33`__ Score the model at five thresholds, from 0.3 to 0.7, with accuracy, precision, recall and the three $F_\beta$ scores of Section 2.6.

In [ ]:
def metrics_at(threshold):
    """Score the validation labels the model gives at one threshold."""
    labels = (y_pred_proba_val >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, labels).ravel()
    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_val, labels),
        "precision": precision_score(y_val, labels, zero_division=0),
        "recall": recall_score(y_val, labels),
        "FPR": fp / (fp + tn),
        "F0.5": fbeta_score(y_val, labels, beta=0.5, zero_division=0),
        "F1": f1_score(y_val, labels, zero_division=0),
        "F2": fbeta_score(y_val, labels, beta=2, zero_division=0),
    }


threshold_rows = [metrics_at(threshold) for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]]
pd.DataFrame(threshold_rows).round(4)

**Each column peaks at a different threshold.** $F_1$ is highest at
**0.4** (0.8601) and $F_2$ at **0.3** (0.9161). $F_{0.5}$ is highest at **0.6**
(0.8392), but **0.7** scores **0.8385**, within a thousandth, so a sweep this
coarse does not locate its maximum.

Which error is expensive depends on **what the prediction is for**. If the model
routes scarce coaching and funding to likely winners, a false positive spends a
season's budget on someone who was never going to place, and $\beta < 1$ is
defensible. If no likely winner may be *overlooked*, the missed athlete is the
expensive error and $\beta > 1$ follows: a threshold near 0.3, with precision
down to **0.7673**, recall bought with precision at a rate you chose.

The ROC and precision-recall curves below show that trade at every threshold.

<a class="anchor" id="roc">

### <font color='#E8800A'>2.8. ROC Curve and AUC Score</font>

</a>

The **ROC (Receiver Operating Characteristic) curve** traces the model across every threshold at once. Each threshold gives one point:
- **Y-axis**: the true positive rate, $TPR = \frac{TP}{TP + FN}$, which is recall;
- **X-axis**: the false positive rate, $FPR = \frac{FP}{FP + TN}$.

The **AUC**, the area under that curve, summarises it in one number: the probability that the model gives a randomly chosen positive a higher score than a randomly chosen negative. 1.0 is a perfect ranking and 0.5 is a coin flip.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html'>sklearn.metrics.roc_auc_score(y_true, y_score, ...)</a>

__Definition:__ <br>
Compute Area Under the Receiver Operating Characteristic Curve (ROC AUC) from prediction scores.

__Interpretation:__ <br>
Best possible score is 1.0. A score of 0.5 indicates random predictions.

__Parameters:__ <br>
_y_true_: Ground truth (correct) target values; <br>
_y_score_: Target scores (probability estimates of the positive class); <br>
...
</div>

__`Step 34`__ Get the **probability scores** for the training rows as well as the validation rows, with `predict_proba`.

In [ ]:
# Get probability predictions for the positive class (column 1)
y_pred_proba_train = log_model.predict_proba(X_train)[:, 1]
y_pred_proba_val = log_model.predict_proba(X_val)[:, 1]

__`Step 35`__ Compute the AUC score for both training and validation sets.

In [ ]:
auc_train = roc_auc_score(y_train, y_pred_proba_train)
auc_train

In [ ]:
auc_val = ...  # <-- CODE HERE
...  # <-- CODE HERE


__`Step 36`__ Plot the ROC curve of the validation rows.

In [ ]:
# Compute ROC curve for validation set
fpr, tpr, thresholds = roc_curve(y_val, y_pred_proba_val)

# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc_score(y_val, y_pred_proba_val):.2f})')
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Random Classifier (AUC = 0.50)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate (Recall)', fontsize=12)
plt.title('Receiver Operating Characteristic (ROC) Curve', fontsize=14)
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

**Understanding the ROC Curve:**
- The dashed diagonal is a model that ranks at random.
- The closer the curve bends towards the top-left corner, the better the ranking; a perfect model passes through (0, 1).

**Advantages of ROC-AUC**:
- Covers every threshold, so it compares models before a threshold has been chosen.
- Does not change with the share of positives, so it compares models across datasets of different balance.

**Disadvantages of ROC-AUC**:
- For the same reason, it can look good on a rare positive class while precision is poor, because a large pool of negatives keeps the false positive rate low.
- It measures ranking, not the quality of the probabilities themselves, and it does not choose a threshold for you.

**When should ROC-AUC be used?**
- To compare models by how well they rank positives above negatives, before the operating threshold is set.

### Using the ROC Curve to Choose a Threshold

The curve also suggests a threshold: the point furthest above the diagonal.

__`Step 37`__ Find the optimal threshold using the **Youden's J statistic** (maximizes TPR - FPR), and print the model's metrics at that threshold.

In [ ]:
# Calculate Youden's J statistic for each threshold
# J = TPR - FPR = Sensitivity - (1 - Specificity)
youdens_j = tpr - fpr

# Find the optimal threshold (maximum J)
optimal_idx = np.argmax(youdens_j)
optimal_threshold_roc = thresholds[optimal_idx]

print(f"Optimal threshold (ROC - Youden's J): {optimal_threshold_roc:.4f}")
print(f"TPR at optimal threshold: {tpr[optimal_idx]:.4f}")
print(f"FPR at optimal threshold: {fpr[optimal_idx]:.4f}\n")
print(pd.Series(metrics_at(optimal_threshold_roc)).round(4).to_string())

__`Step 38`__ Visualize the optimal threshold on the ROC curve.

In [ ]:
# Plot ROC curve with optimal threshold marked
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc_score(y_val, y_pred_proba_val):.2f})')
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Random Classifier')

# Mark the optimal threshold
plt.scatter(fpr[optimal_idx], tpr[optimal_idx], color='red', s=100, 
            label=f'Optimal Threshold = {optimal_threshold_roc:.2f}', zorder=5)

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate (Recall)', fontsize=12)
plt.title('ROC Curve with Optimal Threshold', fontsize=14)
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

<a class="anchor" id="pr-curve">

### <font color='#E8800A'>2.9. Precision-Recall Curve</font>

</a>

The **precision-recall curve** plots precision against recall at every threshold. The ROC curve can look good because true negatives are plentiful; this curve ignores true negatives entirely, so it shows how the model does on the positive class alone.

- **Baseline**: a model that ranks at random has precision equal to the share of positives, at every recall.
- **Better models** hold high precision out to high recall, towards the top-right corner.
- **Average precision**, `average_precision_score`, summarises the curve in one number, as AUC does for ROC.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_recall_curve.html'>sklearn.metrics.precision_recall_curve(y_true, y_score, ...)</a>

__Definition:__ <br>
Compute precision-recall pairs for different probability thresholds.

__Returns:__ <br>
_precision_: Precision values; <br>
_recall_: Recall values; <br>
_thresholds_: Increasing thresholds on the decision function used to compute precision and recall.

__Parameters:__ <br>
_y_true_: Ground truth (correct) target values; <br>
_y_score_: Target scores (probability estimates of the positive class); <br>
...
</div>

__`Step 39`__ Compute and plot the Precision-Recall curve for the validation set.

In [ ]:
# Compute Precision-Recall curve
precision, recall, thresholds_pr = precision_recall_curve(y_val, y_pred_proba_val)

# Calculate the baseline (proportion of positive samples)
baseline = y_val.sum() / len(y_val)

# Plot Precision-Recall curve
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='blue', lw=2, label='PR curve')
plt.axhline(y=baseline, color='gray', linestyle='--', lw=2, label=f'Baseline (Random) = {baseline:.2f}')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curve', fontsize=14)
plt.legend(loc="best")
plt.grid(alpha=0.3)
plt.show()

**Understanding the Precision-Recall Curve:**
- The dashed line is the random baseline, the share of wins in the validation rows.
- Moving right trades precision for recall; the curve is jagged because precision need not fall steadily as the threshold drops.

**Advantages of Precision-Recall Curves**:
- Shows performance on the positive class directly, however many true negatives there are.
- Shows the precision-recall trade-off a threshold will impose.

**Disadvantages of Precision-Recall Curves**:
- Its baseline moves with the share of positives, so curves from datasets of different balance do not compare directly.
- Ignores true negatives, which matter when the negative class is the one of interest.

**When should Precision-Recall Curves be used?**
- When the positive class is rare or is the class you care about, and to pick a threshold from the trade-off it shows.

### Using the Precision-Recall Curve to Choose a Threshold

The curve also suggests a threshold: the one with the highest F1.

__`Step 40`__ Find the optimal threshold by maximizing the **F1 score** (balances precision and recall), and print the model's metrics at that threshold.

In [ ]:
# Calculate F1 score for each threshold
# F1 = 2 * (precision * recall) / (precision + recall)
f1_scores = np.where((precision + recall) == 0, 0, 2 * (precision * recall) / (precision + recall))

# Find the optimal threshold (maximum F1)
optimal_idx_pr = np.argmax(f1_scores)
optimal_threshold_pr = thresholds_pr[optimal_idx_pr]

print(f"Optimal threshold (PR - Max F1): {optimal_threshold_pr:.4f}\n")
print(pd.Series(metrics_at(optimal_threshold_pr)).round(4).to_string())

__`Step 41`__ Visualize the optimal threshold on the Precision-Recall curve.

In [ ]:
# Plot PR curve with optimal threshold marked
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='blue', lw=2, label='PR curve')
plt.axhline(y=baseline, color='gray', linestyle='--', lw=2, label=f'Baseline = {baseline:.2f}')

# Mark the optimal threshold
plt.scatter(recall[optimal_idx_pr], precision[optimal_idx_pr], color='red', s=100, 
            label=f'Optimal Threshold = {optimal_threshold_pr:.2f}\n(Max F1 = {f1_scores[optimal_idx_pr]:.2f})', 
            zorder=5)

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curve with Optimal Threshold', fontsize=14)
plt.legend(loc="best")
plt.grid(alpha=0.3)
plt.show()

__`Step 42`__ Compare the model's metrics at the threshold each method chose.

In [ ]:
# The model's validation metrics at the threshold each method chose.
threshold_methods = pd.DataFrame([
    {"method": "Default", **metrics_at(0.5)},
    {"method": "ROC (Youden's J)", **metrics_at(optimal_threshold_roc)},
    {"method": "PR (Max F1)", **metrics_at(optimal_threshold_pr)},
])
threshold_methods.round(4)

## <font color='#E8800A'>Comparing Metrics</font>

[Back to TOC](#toc)

__`Step 43`__ Put accuracy, precision, recall, F1 and AUC for both sets in one table.

In [ ]:
# Every metric on both sets, computed from the predictions.
scores = {'Accuracy': accuracy_score, 'Precision': precision_score,
          'Recall': recall_score, 'F1': f1_score}
train_scores = {name: score(y_train, y_pred_train) for name, score in scores.items()}
val_scores = {name: score(y_val, y_pred_val) for name, score in scores.items()}
train_scores['AUC'] = roc_auc_score(y_train, y_pred_proba_train)
val_scores['AUC'] = roc_auc_score(y_val, y_pred_proba_val)
classification_metrics = pd.DataFrame({'Train': train_scores, 'Validation': val_scores})
classification_metrics

<a class="anchor" id="classification-comparison">

### <font color='#E8800A'>2.10. How Metrics Can Change Model Selection: A Classification Example</font>

</a>

As in the regression section, a small constructed example shows how the metrics can prefer different models.

__`Step 44`__ Define the true labels and three different sets of predictions representing competing models.

In [ ]:
y_true_cls = np.array([1, 0, 1, 0, 1, 0, 1, 0])

# Model A: balanced performance, few mistakes
preds_model_A_cls = np.array([1, 0, 1, 0, 1, 0, 0, 0])

# Model B: very conservative, rarely predicts positive
preds_model_B_cls = np.array([0, 0, 1, 0, 0, 0, 0, 0])

# Model C: aggressive, predicts many positives
preds_model_C_cls = np.array([1, 1, 1, 0, 1, 1, 1, 0])

__`Step 45`__ Build a helper function that gathers the main classification metrics for any set of predictions.

In [ ]:
def classification_metrics(y_true, y_pred):
    """Return the classification metrics used in this comparison."""
    return [
        accuracy_score(y_true, y_pred),
        precision_score(y_true, y_pred, zero_division=0),
        recall_score(y_true, y_pred, zero_division=0),
        f1_score(y_true, y_pred, zero_division=0),
    ]

__`Step 46`__ Use the helper to compare the models across accuracy, precision, recall, and F1, and display the results in a table.

In [ ]:
classification_comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1'],
    'Model A': classification_metrics(y_true_cls, preds_model_A_cls),
    'Model B': classification_metrics(y_true_cls, preds_model_B_cls),
    'Model C': classification_metrics(y_true_cls, preds_model_C_cls)
})

classification_comparison

__`Step 47`__ Reflect on the table and discuss which model each metric prefers and why.

<div class="alert alert-block alert-success">

### Analysis of the Results

* **Model A** (few mistakes) has the highest accuracy (0.875) and F1 (0.86), with perfect precision and a recall of 0.75.
* **Model B** (conservative) also has perfect precision but finds one positive in four (recall 0.25), so Model A beats it on every metric except precision, where they tie.
* **Model C** (aggressive) finds every positive (recall 1.0) and pays with false alarms (precision 0.67), for an F1 of 0.80.

### What to take from this comparison

Accuracy and F1 prefer A, recall prefers C, and precision cannot separate A from B. Which model is best depends on which error costs more.

</div>

<div class="alert alert-block alert-warning">

**One caveat.** Choosing the metric from the cost of each kind of error, before
you see any scores, makes the choice *defensible*; it does not promise a better
model. The metric that matches your costs may select a model whose headline
figure looks **worse**, and it is still the right model to ship.

</div>

# <font color='#E8800A'>Key takeaways</font> <a class="anchor" id="takeaways"></a>
[Back to TOC](#toc)

What this session established:

1. **A metric is a choice, and it is made before the results arrive**: from the
   cost of each kind of error, not from whichever number came out highest.
2. **MAE and RMSE answer different questions.** Squaring the error lets the
   largest mistakes dominate, so the two rank models differently whenever the
   errors are not all of a similar size.
3. **F1 ignores true negatives.** When the positive class is the majority, as a
   win is here, predicting a win for everyone already scores a high F1, so an F1
   means little without that floor beside it.
4. **A classifier emits probabilities, not labels.** The threshold turns one
   into the other, and moving it trades precision against recall with neither
   the model nor the data changing.
5. **A chosen threshold is reported with its metrics.** A threshold picked on
   the validation rows is also flattered by them, like any other choice made
   there.

<a class="anchor" id="multiclass">

# Optional (Advanced): Multiclass Classification
</a>

Precision, recall and F1 were defined for two classes, one of them positive. With three or more classes each metric is computed per class and then combined, and how they are combined can change the answer more than which metric is chosen. The **Iris dataset** supplies three classes.

__`Step 48`__ Import the Iris dataset and prepare it for multiclass classification.

<div class="alert alert-block alert-info">

**A dataset built to order, for one question only.** Everything above ran on
`champions`, and it will keep running on `champions` for the rest of the course.
This optional section cannot: `Outcome` has exactly two classes, and a multiclass
confusion matrix needs at least three. `load_iris` supplies them.

Nothing measured on Iris joins the course scoreboard; it answers "what does
this metric look like with K classes", not "which model should we ship".

</div>

In [ ]:
from sklearn.datasets import load_iris

# Iris as shipped is 50/50/50 and nearly separable, and on balanced
# classes of equal difficulty every averaging scheme returns the same
# number -- which would make the next three sections identical. Keeping
# one species rare is what gives them something to disagree about.
iris = load_iris()
rare = np.random.RandomState(RANDOM_STATE)
keep = np.sort(np.concatenate([
    rare.choice(np.where(iris.target == 0)[0], 50, replace=False),
    rare.choice(np.where(iris.target == 1)[0], 50, replace=False),
    rare.choice(np.where(iris.target == 2)[0], 10, replace=False),
]))
X_iris = iris.data[keep]
y_iris = iris.target[keep]

print("rows per species:", np.bincount(y_iris),
      "->", dict(zip(iris.target_names, np.bincount(y_iris))))

__`Step 49`__ Split the data and train a multiclass logistic regression model.

In [ ]:
# Split into train and test sets
X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=RANDOM_STATE, stratify=y_iris
)

# Train a multiclass logistic regression
iris_model = LogisticRegression(max_iter=200, random_state=RANDOM_STATE)
iris_model.fit(X_train_iris, y_train_iris)

# Make predictions
y_pred_iris = iris_model.predict(X_test_iris)

<a class="anchor" id="multiclass-confusion">

### <font color='#E8800A'>3.1. Multiclass Confusion Matrix</font>
</a>

With K classes the confusion matrix is **K × K**: rows are the actual classes and columns the predicted ones.

|                  | Predicted Class 0 | Predicted Class 1 | Predicted Class 2 |
|------------------|-------------------|-------------------|-------------------|
| **Actual Class 0** | Correct (TP₀)     | Misclassified     | Misclassified     |
| **Actual Class 1** | Misclassified     | Correct (TP₁)     | Misclassified     |
| **Actual Class 2** | Misclassified     | Misclassified     | Correct (TP₂)     |

The diagonal holds the correct predictions, and each off-diagonal cell counts one way two classes are confused.

__`Step 50`__ Create and visualize the multiclass confusion matrix for the Iris predictions.

In [ ]:
# Create confusion matrix
cm_iris = confusion_matrix(y_test_iris, y_pred_iris)

# Visualize with a heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(cm_iris, annot=True, fmt='d', cmap='Blues', 
            xticklabels=iris.target_names,
            yticklabels=iris.target_names,
            cbar=True)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Multiclass Confusion Matrix - Iris Dataset', fontsize=14)
plt.show()

**Advantages of the Multiclass Confusion Matrix**:
- Shows which classes are confused with which, not only how often the model is wrong.
- Every per-class precision, recall and F1 is computed from it.

**Disadvantages of the Multiclass Confusion Matrix**:
- With many classes it becomes hard to read, and it is not a single number.

**When should the Multiclass Confusion Matrix be used?**
- As the first step of any multiclass evaluation.

<a class="anchor" id="macro">

### <font color='#E8800A'>3.2. Macro-Averaged Metrics</font>
</a>

Each class is scored as a binary problem, **One-vs-Rest**: that class is positive and every other class negative, so its TP is its diagonal cell, its FP the rest of its column and its FN the rest of its row. **Macro averaging** takes the simple, unweighted mean of the per-class scores:

$$
\text{Macro Metric} = \frac{1}{K} \sum_{i=1}^{K} \text{Metric}_i
$$

__`Step 51`__ Calculate macro-averaged precision, recall, and F1 score.

In [ ]:
# Calculate macro-averaged metrics
macro_precision = precision_score(y_test_iris, y_pred_iris, average='macro')
macro_recall = recall_score(y_test_iris, y_pred_iris, average='macro')
macro_f1 = f1_score(y_test_iris, y_pred_iris, average='macro')

print("Macro-Averaged Metrics:")
print(f"Precision: {macro_precision:.4f}")
print(f"Recall:    {macro_recall:.4f}")
print(f"F1 Score:  {macro_f1:.4f}")

# Also show per-class metrics to understand the macro average
print("\nPer-Class Metrics (used to calculate macro average):")
prec_per_class = precision_score(y_test_iris, y_pred_iris, average=None)
rec_per_class = recall_score(y_test_iris, y_pred_iris, average=None)
f1_per_class = f1_score(y_test_iris, y_pred_iris, average=None)

for i, class_name in enumerate(iris.target_names):
    print(f"{class_name:12} - Precision: {prec_per_class[i]:.4f}, Recall: {rec_per_class[i]:.4f}, F1: {f1_per_class[i]:.4f}")

print(f"\nMacro Precision = ({prec_per_class[0]:.4f} + {prec_per_class[1]:.4f} + {prec_per_class[2]:.4f}) / 3 = {macro_precision:.4f}")

**Advantages of Macro Averaging**:
- Every class counts equally, so poor performance on a rare class shows.

**Disadvantages of Macro Averaging**:
- A class with ten rows moves the score as much as one with a thousand, so it is noisy when a class is small.

**When should Macro Averaging be used?**
- When every class matters equally, however rare it is.

<a class="anchor" id="weighted">

### <font color='#E8800A'>3.3. Weighted-Averaged Metrics</font>
</a>

**Weighted averaging** weights each class's score by its share of the true rows, $w_i = n_i / N$:

$$
\text{Weighted Metric} = \sum_{i=1}^{K} w_i \times \text{Metric}_i
$$

__`Step 52`__ Calculate weighted-averaged precision, recall, and F1 score.

In [ ]:
# Calculate weighted-averaged metrics
weighted_precision = precision_score(y_test_iris, y_pred_iris, average='weighted')
weighted_recall = recall_score(y_test_iris, y_pred_iris, average='weighted')
weighted_f1 = f1_score(y_test_iris, y_pred_iris, average='weighted')

print("Weighted-Averaged Metrics:")
print(f"Precision: {weighted_precision:.4f}")
print(f"Recall:    {weighted_recall:.4f}")
print(f"F1 Score:  {weighted_f1:.4f}")

# Show the calculation
print("\nHow weighted average is calculated:")
class_counts = np.bincount(y_test_iris)
total_samples = len(y_test_iris)

weighted_prec_manual = 0
for i, class_name in enumerate(iris.target_names):
    weight = class_counts[i] / total_samples
    contribution = weight * prec_per_class[i]
    weighted_prec_manual += contribution
    print(f"{class_name:12} - {class_counts[i]} samples ({weight:.3f} weight) × {prec_per_class[i]:.4f} precision = {contribution:.4f}")

print(f"\nWeighted Precision = {weighted_prec_manual:.4f}")

# Compare with macro
print(f"\nComparison:")
print(f"Macro Precision:    {macro_precision:.4f} (equal weight to all classes)")
print(f"Weighted Precision: {weighted_precision:.4f} (weighted by class frequency)")

**Advantages of Weighted Averaging**:
- Each class counts in proportion to its size, so the score describes the typical row.

**Disadvantages of Weighted Averaging**:
- A rare class barely moves it, so poor performance there can hide.

**When should Weighted Averaging be used?**
- When classes matter in proportion to how often they occur.

<a class="anchor" id="micro">

### <font color='#E8800A'>3.4. Micro-Averaged Metrics</font>
</a>

**Micro averaging** pools the true positives, false positives and false negatives of every class before computing the metric:

$$
\text{Micro Precision} = \frac{\sum_{i} TP_i}{\sum_{i} (TP_i + FP_i)} \qquad \text{Micro Recall} = \frac{\sum_{i} TP_i}{\sum_{i} (TP_i + FN_i)}
$$

In a single-label problem every prediction is one TP or one FP, and every row one TP or one FN, so **micro precision, micro recall, micro F1 and accuracy are all equal**.

__`Step 53`__ Calculate micro-averaged metrics and verify they equal accuracy.

In [ ]:
# Calculate micro-averaged metrics
micro_precision = precision_score(y_test_iris, y_pred_iris, average='micro')
micro_recall = recall_score(y_test_iris, y_pred_iris, average='micro')
micro_f1 = f1_score(y_test_iris, y_pred_iris, average='micro')

# Calculate accuracy for comparison
accuracy = accuracy_score(y_test_iris, y_pred_iris)

print("Micro-Averaged Metrics:")
print(f"Precision: {micro_precision:.4f}")
print(f"Recall:    {micro_recall:.4f}")
print(f"F1 Score:  {micro_f1:.4f}")
print(f"\nAccuracy:  {accuracy:.4f}")

**Advantages of Micro Averaging**:
- Pools every prediction, so each row counts once.

**Disadvantages of Micro Averaging**:
- In a single-label multiclass problem it equals accuracy, so it adds nothing accuracy does not already say, and the largest classes dominate it.

**When should Micro Averaging be used?**
- Beside the macro and weighted averages, to show how much the class sizes move the score.

## <font color='#E8800A'>Comparing All Averaging Methods</font>

__`Step 54`__ Create a comparison table of all averaging methods.

[Back to TOC](#toc)

In [ ]:
# Every averaging method on every metric. Accuracy is not averaged over
# classes, so it is the same in every row, and it equals the micro averages.
comparison_df = pd.DataFrame(
    {
        "Precision": [macro_precision, weighted_precision, micro_precision],
        "Recall": [macro_recall, weighted_recall, micro_recall],
        "F1 Score": [macro_f1, weighted_f1, micro_f1],
        "Accuracy": [accuracy, accuracy, accuracy],
    },
    index=["Macro", "Weighted", "Micro"],
)
print("Comparison of All Averaging Methods:")
print(comparison_df.round(4).to_string())

__`Step 55`__ Use `classification_report` to see all metrics in one comprehensive view.

In [ ]:
# The classification_report provides everything in one place
print(classification_report(y_test_iris, y_pred_iris, target_names=iris.target_names))

### Choosing an averaging method

<div class="alert alert-block alert-success">

```
Do all classes have equal importance?
├─ YES → Use MACRO averaging
└─ NO → Is class frequency aligned with importance?
    ├─ YES → Use WEIGHTED averaging
    └─ NO → Consider MACRO or define custom weights
```

Start from the **confusion matrix** to see where the errors are, read the **per-class** metrics (`average=None`) to find the classes the model struggles with, and then choose the average from what the problem needs. **Micro** averaging equals accuracy here, so it is a reference rather than a choice. `classification_report` prints all of them at once.

</div>

Sources: <br>
[scikit-learn User Guide: Metrics and scoring](https://scikit-learn.org/stable/modules/model_evaluation.html) <br>
[Regression metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics) <br>
[Classification metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics) <br>
[Tuning the decision threshold for class prediction](https://scikit-learn.org/stable/modules/classification_threshold.html)